In [1]:
# System-level: Ollama binary + models
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull llama3.2
!ollama pull mxbai-embed-large

# Python packages
!pip install -q langchain langchain-community langchain-ollama langchain-chroma chromadb pypdf langchain-text-splitters pandas

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.




In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

embeddings = OllamaEmbeddings(model="mxbai-embed-large")

db_location = "./chroma_langchain_db"
directory_path = "/content/policies"
add_documents = not os.path.exists(db_location)
print(add_documents) 
if add_documents:
    all_documents = []   
    # Iterate through all files in the directory
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        
        # Skip directories
        if os.path.isdir(file_path):
            continue
            
        # Process PDFs (Unstructured narrative data)
        if filename.endswith('.pdf'):
            print(f"Loading PDF: {filename}")
            loader = PyPDFLoader(file_path)
            all_documents.extend(loader.load())
            
        # Process CSVs (Structured tabular data)
        elif filename.endswith('.csv'):
            print(f"Loading CSV: {filename}")
            # CSVLoader converts each row into a self-contained text document
            loader = CSVLoader(file_path=file_path)
            all_documents.extend(loader.load())

    raw_docs = all_documents
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    documents = splitter.split_documents(raw_docs)
    ids = [str(i) for i in range(len(documents))]

vector_store = Chroma.from_documents(
    documents=documents, 
    embedding=embeddings,
    persist_directory=db_location
)

#if add_documents:
   # vector_store.add_documents(documents=documents, ids=ids)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

/tmp/ipykernel_11398/900057119.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader, CSVLoader


True
Loading PDF: PolicyDocument-Jeevan_Utsav.pdf
Loading PDF: Policy_Document_Jeevan_Utsav_Single Premium.pdf
Loading PDF: Policy_doc_Jeevan_Umang.pdf
Loading PDF: Jeevan_Utsav_Sales_Brochure.pdf
Loading PDF: policy_names.pdf
Loading PDF: CIS _Jeevan_Umang.pdf
Loading PDF: money_back_plan_details.pdf
Loading PDF: LIC_Jeevan_Utsav _Single_Premium_Sales_Brochure.pdf
Loading PDF: LIC_Jeevan_Umang_Sales_Brochure.pdf
Loading PDF: CIS_Jeevan_Utsav_Single_Premium.pdf
Loading PDF: CIS _jeevan_utsav.pdf


In [3]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

model = OllamaLLM(model="llama3.2")

template = """
You are a expert in answering user queries related to life insurance policies.
Use the below context and answer the user queries as much as possible without deviating from
the information mentioned in the context.

Context: {context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

while True:
    print("\n\n-------------------------------")
    question = input("Ask your question (q to quit): ")
    print("\n\n")
    if question == "q":
        break

    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    result = chain.invoke({"context": context, "question": question})
    print("PULLED CONTEXT: ", context)
    print("\n\nCONCLUSION: ", result)



-------------------------------



PULLED CONTEXT:  Here is a list of policies/plans included in the current context: 
1. Jeevan Umang 
2. Jeevan Utsav 
3. Jeevan Utsav Single Premium

Policy and the benefits available under the Policy. 
 
Some of our plans have certain options (including rider(s)) available under them. It is important that 
the options, if any, available under this plan and mentioned in the Policy Document are noted 
carefully as it will be helpful to you in case you decide to exercise any of the available options. It is 
also essential to note that such options, if available and mentioned in the document of this plan has 
to be exercised in the right manner and during the stipulated time limit as prescribed herein. 
 
Free Look Period 
 
We would request you to go through the terms and conditions of the Policy and in case you 
disagree with any of the terms and conditions, you may return the Policy within a period of 30 days 
from the date of receipt of the electro